In [36]:
import json
import os
from datetime import datetime

import pandas as pd
import janitor
import requests
from tqdm.notebook import tqdm

today_str = datetime.today().strftime('%Y-%m-%d')
today_str

'2026-03-18'

In [39]:
df_ts = (
    pd.read_csv("../../get_metrics/output/repo_stars_timeseries.csv")
    .drop_duplicates("fileslug")
    .select_columns(["fileslug", "treated2"])
    .assign(slug=lambda df: df["fileslug"].str.replace("_", "/", n=1))
)

df_ts.head(3)

/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)


,fileslug,treated2,slug
0,0riion_py-sls-lambda-toolkit,2,0riion/py-sls-lambda-toolkit
30,0x9900_sonde2kml,0,0x9900/sonde2kml
60,1azybug_chinese-webtext-spider,0,1azybug/chinese-webtext-spider


In [25]:
with open("../input/secrets.txt") as f:
    GITHUB_TOKEN = f.read().strip()
    

headers = {"Authorization": f"token {GITHUB_TOKEN}"}

output_dir = f"../output/cache_payload/{today_str}"
os.makedirs(output_dir, exist_ok=True)

for slug in tqdm(df_ts.query("treated2 == 2")["slug"].tolist()):
    fileslug = slug.replace("/", "_")
    out_path = f"{output_dir}/{today_str}_{fileslug}.json"

    if os.path.exists(out_path):
        continue

    url = f"https://api.github.com/repos/{slug}"
    response = requests.get(url, headers=headers, timeout=15)
    rc = response.status_code

    if rc not in (200, 404, 301):
        print(f"{slug}: {rc}")
    
    if rc == 200:
        payload = {**response.json(), "_status": rc} 
    else:
        payload = {"_status": rc}

    with open(out_path, "w") as f:
        json.dump(payload, f)    

  0%|          | 0/25 [00:00<?, ?it/s]

In [34]:
records = []
for fname in os.listdir(output_dir):
    if not fname.endswith(".json"):
        continue
    fileslug = fname.replace(f"{today_str}_", "").replace(".json", "")
    with open(f"{output_dir}/{fname}") as f:
        payload = json.load(f)
    records.append({
        "fileslug": fileslug,
        "stars_today": payload.get("stargazers_count", None),
        "status": payload.get("_status"),
    })

df_today = (
    pd.DataFrame(records)
    .merge(df_ts[["fileslug", "treated2"]], on="fileslug", how="left", validate="1:1")
)

print(df_today["status"].value_counts())
df_today

status
200    23
404     2
Name: count, dtype: int64


,fileslug,stars_today,status,treated2
0,cancan101_graphql-db-api,52.0,200,2
1,scipion-em_scipion-em-isonet,20.0,200,2
2,deepmind_mujoco,12402.0,200,2
3,bond-anton_bdpotentiometer,21.0,200,2
4,tlouarn_pyopenfigi,23.0,200,2
5,mrq-andras_asciicli,NaN,404,2
6,kotlasaicharanreddy_pravega-client-rust,19.0,200,2
7,chaimain_asgardpy,22.0,200,2
8,oca_purchase-workflow,272.0,200,2
9,thejcannon_botocore-a-la-carte,25.0,200,2


In [38]:
df_post = (
    pd.read_csv("../../get_metrics/output/repo_stars_timeseries.csv")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .query("date == '2023-05-21'")
    .select_columns(["fileslug", "stars"])
    .rename_column("stars", "stars_post_treat")
)

df = df_today.query("status == 200").merge(df_post, on="fileslug", how="left")
df[["fileslug", "stars_post_treat", "stars_today"]].sort_values("fileslug")

/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)
/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.rename` instead.
  return method(self._obj, *args, **kwargs)


,fileslug,stars_post_treat,stars_today
21,0riion_py-sls-lambda-toolkit,69,20.0
10,andrew-dickinson_bird-ospf-link-db-parser,69,20.0
14,avialxee_scoobi,69,20.0
3,bond-anton_bdpotentiometer,69,21.0
0,cancan101_graphql-db-api,86,52.0
6,chaimain_asgardpy,68,22.0
22,chaturap_bmkg-latest-earthquake,69,20.0
2,deepmind_mujoco,100,12402.0
17,firebase_firebase-functions-python,100,165.0
20,hansalemaos_locate_pixelcolor_cythonsingle,68,23.0


In [32]:
df["stars_change"] = df["stars_today"] - df["stars_post_treat"]

df[["fileslug", "stars_post_treat", "stars_today", "stars_change"]].describe()

,stars_post_treat,stars_today,stars_change
count,23.000000,23.000000,23.000000
mean,74.000000,806.652174,732.652174
std,16.684233,2750.511020,2742.808193
min,20.000000,19.000000,-51.000000
25%,69.000000,20.000000,-49.000000
50%,70.000000,22.000000,-46.000000
75%,76.500000,41.000000,-34.500000
max,100.000000,12402.000000,12302.000000
